# Exhaustive LangChain Agents
## Tools, Retries, and Fallbacks

## 1. Install Dependencies

In [ ]:
!pip install langchain langchain-groq duckduckgo-search

## 2. Imports

In [ ]:
import os
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_groq import ChatGroq

## 3. Define Tools (Internet Search & Custom Functions)

In [ ]:
# Tool 1: Built-in Internet Search (DuckDuckGo)
search_tool = DuckDuckGoSearchRun()

# Tool 2: Custom Python Function
def calculate_taxes(income: str) -> str:
    return str(float(income) * 0.2)

tax_tool = Tool(
    name="Tax Calculator",
    func=calculate_taxes,
    description="Useful for calculating taxes on a given numeric income."
)

# Group them together
tools = [search_tool, tax_tool]

## 4. Initialize Models with Retries and Fallbacks

In [ ]:
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'

# PRIMARY MODEL: Large reasoning model.
# Retries: If the API fails or rate-limits, try again up to 3 times.
primary_llm = ChatGroq(
    model='llama-3.3-70b-versatile', 
    temperature=0, 
    max_retries=3
)

# FALLBACK MODEL: Smaller, faster model.
# If the primary model completely crashes after 3 retries, use this one instead.
fallback_llm = ChatGroq(
    model='llama3-8b-8192', 
    temperature=0
)

# Combine them: "Try Primary first, if it fails, use Fallback"
resilient_llm = primary_llm.with_fallbacks([fallback_llm])

## 5. Encapsulated Resilient Agent Function

In [ ]:
def run_resilient_agent(query: str):
    # Initialize the agent executor
    agent_executor = initialize_agent(
        tools=tools,
        llm=resilient_llm,                          # Uses our Retry + Fallback model
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        handle_parsing_errors=True,                 # Tool Retries: If the LLM hallucinates a tool format, it self-corrects!
        verbose=True
    )
    
    return agent_executor.run(query)

## 6. Test the Agent

In [ ]:
# Test 1: It should use DuckDuckGo to find current info
search_answer = run_resilient_agent("Who won the Super Bowl in 2024?")
print("Search Result:", search_answer)

# Test 2: It should use the Tax Calculator
tax_answer = run_resilient_agent("If I make 100000 dollars, what are my taxes?")
print("Tax Result:", tax_answer)